# 🏀 TakeMeter — r/nba Discourse Classifier
**Labels:** `analysis` | `hot_take` | `reaction`

**Runtime required:** T4 GPU → Runtime → Change runtime type → T4 GPU

## Section 1 — Setup & Data Loading

In [1]:
# Install dependencies
!pip install transformers datasets scikit-learn groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
import os
from google.colab import files

# ── Label map ─────────────────────────────────────────────
LABEL2ID = {"analysis": 0, "hot_take": 1, "reaction": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print("Label map:", LABEL2ID)

# ── Upload your data.csv ───────────────────────────────────
print("\nUpload your data.csv file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
df = df[['text', 'label']].dropna()
df = df[df['label'].isin(LABEL2ID.keys())].reset_index(drop=True)
df['label_id'] = df['label'].map(LABEL2ID)

print(f"\nLoaded {len(df)} examples")
print("\nLabel distribution:")
print(df['label'].value_counts())

Label map: {'analysis': 0, 'hot_take': 1, 'reaction': 2}

Upload your data.csv file:


Saving data.csv to data.csv

Loaded 201 examples

Label distribution:
label
analysis    67
hot_take    67
reaction    67
Name: count, dtype: int64


## Section 2 — Train / Val / Test Split & Tokenization

In [3]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 70 / 15 / 15 split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nTrain distribution:")
print(train_df['label'].value_counts())
print("\nTest distribution:")
print(test_df['label'].value_counts())

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

def to_hf_dataset(dataframe):
    ds = Dataset.from_dict({
        'text': dataframe['text'].tolist(),
        'labels': dataframe['label_id'].tolist()
    })
    return ds.map(tokenize, batched=True)

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

print("\nTokenization complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train: 140 | Val: 30 | Test: 31

Train distribution:
label
analysis    47
hot_take    47
reaction    46
Name: count, dtype: int64

Test distribution:
label
reaction    11
hot_take    10
analysis    10
Name: count, dtype: int64


Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]


Tokenization complete.


In [5]:
import transformers
print(transformers.__version__)

5.12.0


## Section 3 — Fine-Tune DistilBERT

In [6]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro')
    }

# ── Hyperparameters ────────────────────────────────────────
# learning_rate=2e-5: standard for DistilBERT fine-tuning;
# lower values (1e-5) are safer but slower to converge on small data.
# num_train_epochs=4: one extra epoch vs default helps on 140 training examples.
# batch_size=16: fits T4 memory comfortably with max_length=128.

training_args = TrainingArguments(
    output_dir="./takemeter_model",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",      # <-- changed
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning... (~5-15 minutes on T4 GPU)")
trainer.train()
print("\nTraining complete!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning... (~5-15 minutes on T4 GPU)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.010603,0.600000,0.487386
2,1.076581,0.875646,0.800000,0.771284
3,0.933031,0.782655,0.766667,0.723543
4,0.813776,0.749467,0.833333,0.815108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!


## Section 4 — Evaluate Fine-Tuned Model

In [7]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)

# ── Run predictions on test set ────────────────────────────
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# ── Confidence scores ──────────────────────────────────────
import torch
logits_tensor = torch.tensor(predictions.predictions)
probs = torch.softmax(logits_tensor, dim=-1).numpy()
confidences = probs.max(axis=-1)

# ── Metrics ────────────────────────────────────────────────
finetuned_accuracy = accuracy_score(true_labels, preds)
label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print(f"\n{'='*50}")
print(f"FINE-TUNED MODEL — Test Set Results")
print(f"{'='*50}")
print(f"Overall Accuracy: {finetuned_accuracy:.4f} ({finetuned_accuracy*100:.1f}%)")
print(f"\nPer-class Report:")
report = classification_report(true_labels, preds, target_names=label_names, output_dict=True)
print(classification_report(true_labels, preds, target_names=label_names))

# ── Confusion matrix ───────────────────────────────────────
cm = confusion_matrix(true_labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('TakeMeter — Fine-Tuned Model Confusion Matrix', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

# ── Wrong predictions ──────────────────────────────────────
test_texts = test_df['text'].tolist()
test_true  = test_df['label'].tolist()
test_pred  = [ID2LABEL[p] for p in preds]
test_conf  = confidences.tolist()

wrong = [
    {"text": test_texts[i], "true": test_true[i],
     "predicted": test_pred[i], "confidence": round(test_conf[i], 4)}
    for i in range(len(preds)) if test_pred[i] != test_true[i]
]

print(f"\nWrong predictions: {len(wrong)} / {len(preds)}")
print("\nFirst 5 wrong predictions:")
for i, w in enumerate(wrong[:5]):
    print(f"\n[{i+1}] True: {w['true']} | Predicted: {w['predicted']} | Confidence: {w['confidence']}")
    print(f"     Text: {w['text'][:120]}...")

# Store for Section 6
finetuned_report = report
finetuned_wrong  = wrong
finetuned_cm     = cm.tolist()
finetuned_preds  = list(zip(test_texts, test_true, test_pred, test_conf))


FINE-TUNED MODEL — Test Set Results
Overall Accuracy: 0.8387 (83.9%)

Per-class Report:
              precision    recall  f1-score   support

    analysis       0.75      0.90      0.82        10
    hot_take       0.88      0.70      0.78        10
    reaction       0.91      0.91      0.91        11

    accuracy                           0.84        31
   macro avg       0.84      0.84      0.84        31
weighted avg       0.85      0.84      0.84        31

Saved: confusion_matrix.png

Wrong predictions: 5 / 31

First 5 wrong predictions:

[1] True: hot_take | Predicted: analysis | Confidence: 0.39
     Text: The entire Eastern Conference is fraudulent this year. Any team from the West would win the East by ten games. Complete ...

[2] True: analysis | Predicted: reaction | Confidence: 0.3965
     Text: San Antonio is doing everything right in the rebuild. They are not rushing Wembanyama, developing the supporting cast ar...

[3] True: reaction | Predicted: hot_take | Confidenc

## Section 5 — Groq Zero-Shot Baseline

In [8]:
from groq import Groq
from sklearn.metrics import classification_report, accuracy_score

# ── Add your Groq API key ──────────────────────────────────
# Option A (recommended): Colab Secrets → add GROQ_API_KEY
# Option B: paste directly (do NOT commit to GitHub)

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("Loaded API key from Colab Secrets")
except Exception:
    GROQ_API_KEY = ""  # fallback
    print("Using hardcoded key — do not commit this to GitHub")

client = Groq(api_key=GROQ_API_KEY)

CLASSIFICATION_PROMPT = """You are labeling r/nba Reddit posts for a text classification dataset.
Label the post with EXACTLY ONE of these three labels:

analysis — Makes a structured argument backed by specific stats, historical comparisons,
or tactical observations. Evidence is concrete and verifiable. Removing the opinion framing
would still leave a real argument standing.

hot_take — A bold, confident opinion stated without supporting evidence. Asserts a claim
that generalizes beyond a single game or event. The claim might be true but the post argues
nothing — it just states.

reaction — An immediate emotional response to a specific game, play, or news event.
Primarily expressing a feeling. Little to no argument.

DECISION RULES:
- One cherry-picked stat + opinion framing → hot_take
- Emotional + tied to specific event + no generalizing claim → reaction
- Emotional + generalizing claim about a player/team → hot_take
- Remove opinion words and a verifiable argument remains → analysis

Respond with ONLY the label name. Nothing else. No explanation.
Valid responses: analysis, hot_take, reaction
"""

def classify_groq(text):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": CLASSIFICATION_PROMPT},
                {"role": "user", "content": f"Post: {text}"}
            ],
            max_tokens=10,
            temperature=0
        )
        label = response.choices[0].message.content.strip().lower()
        label = label.replace('"', '').replace("'", '').strip()
        if label not in LABEL2ID:
            return None
        return label
    except Exception as e:
        print(f"Error: {e}")
        return None

print(f"Running Groq baseline on {len(test_texts)} test examples...")
print("This may take 2-3 minutes due to API rate limits.\n")

import time
groq_preds = []
unparseable = 0

for i, text in enumerate(test_texts):
    pred = classify_groq(text)
    if pred is None:
        unparseable += 1
        pred = "hot_take"  # fallback to majority class
    groq_preds.append(pred)
    if (i + 1) % 10 == 0:
        print(f"  Classified {i+1}/{len(test_texts)}...")
    time.sleep(0.3)  # avoid rate limit

groq_true_ids  = [LABEL2ID[l] for l in test_true]
groq_pred_ids  = [LABEL2ID[l] for l in groq_preds]

baseline_accuracy = accuracy_score(groq_true_ids, groq_pred_ids)
baseline_report   = classification_report(
    groq_true_ids, groq_pred_ids,
    target_names=label_names,
    output_dict=True
)

print(f"\n{'='*50}")
print(f"GROQ ZERO-SHOT BASELINE — Test Set Results")
print(f"{'='*50}")
print(f"Overall Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.1f}%)")
print(f"Unparseable responses: {unparseable}")
print(f"\nPer-class Report:")
print(classification_report(groq_true_ids, groq_pred_ids, target_names=label_names))

Using hardcoded key — do not commit this to GitHub
Running Groq baseline on 31 test examples...
This may take 2-3 minutes due to API rate limits.

  Classified 10/31...
  Classified 20/31...
  Classified 30/31...

GROQ ZERO-SHOT BASELINE — Test Set Results
Overall Accuracy: 0.9032 (90.3%)
Unparseable responses: 0

Per-class Report:
              precision    recall  f1-score   support

    analysis       1.00      0.80      0.89        10
    hot_take       0.77      1.00      0.87        10
    reaction       1.00      0.91      0.95        11

    accuracy                           0.90        31
   macro avg       0.92      0.90      0.90        31
weighted avg       0.93      0.90      0.91        31



## Section 6 — Side-by-Side Comparison & Export

In [9]:
print(f"{'='*55}")
print(f"{'MODEL':<25} {'ACCURACY':>10} {'F1 MACRO':>10}")
print(f"{'-'*55}")
print(f"{'Groq Zero-Shot Baseline':<25} {baseline_accuracy*100:>9.1f}% {baseline_report['macro avg']['f1-score']*100:>9.1f}%")
print(f"{'DistilBERT Fine-Tuned':<25} {finetuned_accuracy*100:>9.1f}% {finetuned_report['macro avg']['f1-score']*100:>9.1f}%")
print(f"{'='*55}")
delta = (finetuned_accuracy - baseline_accuracy) * 100
print(f"\nImprovement from fine-tuning: {delta:+.1f} percentage points")

print("\nPer-class F1 comparison:")
print(f"{'Label':<12} {'Baseline F1':>12} {'Fine-Tuned F1':>14}")
print("-" * 40)
for lbl in label_names:
    b_f1 = baseline_report.get(lbl, {}).get('f1-score', 0)
    f_f1 = finetuned_report.get(lbl, {}).get('f1-score', 0)
    print(f"{lbl:<12} {b_f1*100:>11.1f}% {f_f1*100:>13.1f}%")

# ── Sample classifications table ───────────────────────────
print("\n" + "="*55)
print("SAMPLE CLASSIFICATIONS (Fine-Tuned Model)")
print("="*55)
sample_indices = list(range(0, min(5, len(finetuned_preds))))
for idx in sample_indices:
    text, true, pred, conf = finetuned_preds[idx]
    correct = "✅" if true == pred else "❌"
    print(f"\n{correct} True: {true:<10} Predicted: {pred:<10} Confidence: {conf:.2%}")
    print(f"   Text: {text[:100]}...")

# ── Export evaluation_results.json ─────────────────────────
results = {
    "baseline": {
        "model": "llama-3.3-70b-versatile (zero-shot)",
        "accuracy": round(baseline_accuracy, 4),
        "per_class": {
            lbl: {
                "precision": round(baseline_report.get(lbl, {}).get('precision', 0), 4),
                "recall":    round(baseline_report.get(lbl, {}).get('recall', 0), 4),
                "f1":        round(baseline_report.get(lbl, {}).get('f1-score', 0), 4)
            } for lbl in label_names
        },
        "macro_f1": round(baseline_report['macro avg']['f1-score'], 4)
    },
    "finetuned": {
        "model": "distilbert-base-uncased (fine-tuned)",
        "accuracy": round(finetuned_accuracy, 4),
        "per_class": {
            lbl: {
                "precision": round(finetuned_report.get(lbl, {}).get('precision', 0), 4),
                "recall":    round(finetuned_report.get(lbl, {}).get('recall', 0), 4),
                "f1":        round(finetuned_report.get(lbl, {}).get('f1-score', 0), 4)
            } for lbl in label_names
        },
        "macro_f1": round(finetuned_report['macro avg']['f1-score'], 4),
        "confusion_matrix": {
            "labels": label_names,
            "matrix": finetuned_cm
        }
    },
    "wrong_predictions": finetuned_wrong,
    "sample_classifications": [
        {"text": t, "true": tr, "predicted": pr, "confidence": round(c, 4)}
        for t, tr, pr, c in finetuned_preds[:5]
    ]
}

with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Saved: evaluation_results.json")
print("✅ Saved: confusion_matrix.png")
print("\nDownload both files from the Files panel (left sidebar) → right-click → Download")

MODEL                       ACCURACY   F1 MACRO
-------------------------------------------------------
Groq Zero-Shot Baseline        90.3%      90.4%
DistilBERT Fine-Tuned          83.9%      83.5%

Improvement from fine-tuning: -6.5 percentage points

Per-class F1 comparison:
Label         Baseline F1  Fine-Tuned F1
----------------------------------------
analysis            88.9%          81.8%
hot_take            87.0%          77.8%
reaction            95.2%          90.9%

SAMPLE CLASSIFICATIONS (Fine-Tuned Model)

❌ True: hot_take   Predicted: analysis   Confidence: 39.00%
   Text: The entire Eastern Conference is fraudulent this year. Any team from the West would win the East by ...

✅ True: hot_take   Predicted: hot_take   Confidence: 41.57%
   Text: Whoever is running the Wizards front office should never work in basketball again. The decision maki...

✅ True: hot_take   Predicted: hot_take   Confidence: 38.93%
   Text: Embiid will never win a championship. Too injury prone

## Section 7 — Deployed Interface (Gradio)

In [10]:
!pip install gradio -q

import gradio as gr
import torch

def classify_post(text):
    if not text.strip():
        return "Please enter a post.", {}

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs  = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
    pred   = ID2LABEL[int(np.argmax(probs))]
    scores = {ID2LABEL[i]: float(round(probs[i], 4)) for i in range(NUM_LABELS)}

    label_descriptions = {
        "analysis": "📊 Analysis — structured argument with verifiable evidence",
        "hot_take": "🔥 Hot Take — bold opinion with no supporting evidence",
        "reaction": "😤 Reaction — emotional response to a specific moment"
    }

    result = f"**Prediction: {label_descriptions[pred]}**\n\nConfidence: {max(probs)*100:.1f}%"
    return result, scores

examples = [
    ["Jokic's PER of 31.1 is the highest ever recorded and he shoots 57% from the field. The most efficient big in NBA history."],
    ["LeBron is the GOAT and it's not even close. Jordan wouldn't survive in today's game."],
    ["CURRY JUST HIT THAT FROM HALF COURT NO WAY 😭😭😭 I am actually shaking right now"],
    ["The Celtics defensive rating drops 8 points when Tatum guards the perimeter. Teams are hunting that mismatch every possession."],
    ["This team is going to give me a heart attack I swear. Down 15 again in the fourth."]
]

with gr.Blocks(title="TakeMeter", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏀 TakeMeter — r/nba Discourse Classifier")
    gr.Markdown("Classifies r/nba posts as **analysis**, **hot_take**, or **reaction**")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Enter an r/nba post or comment",
                placeholder="e.g. LeBron is the GOAT and it's not even close...",
                lines=4
            )
            classify_btn = gr.Button("Classify", variant="primary")

        with gr.Column(scale=1):
            result_output = gr.Markdown(label="Result")
            scores_output = gr.Label(label="Confidence Scores", num_top_classes=3)

    gr.Examples(examples=examples, inputs=text_input)

    classify_btn.click(
        fn=classify_post,
        inputs=text_input,
        outputs=[result_output, scores_output]
    )

demo.launch(share=True, debug=False)
print("\n✅ Interface running. Use the public URL above to demo it.")

/tmp/ipykernel_2066/3757362288.py:45: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="TakeMeter", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30ebd44bd11944ddca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Interface running. Use the public URL above to demo it.
